## 3.4 Alternative synthetic targets


In [ ]:
new_target_definitions = {
    "Balanced Index Tracking": {
        "MXWD Index": 0.60,
        "LEGATRUU Index": 0.40
    },
    "UCITS Clone": {
        "HFRXGL Index": 0.50,
        "MXWD Index": 0.25,
        "LEGATRUU Index": 0.25
    },
    "Risk Management Target": {
        "HFRXGL Index": 0.40,
        "MXWD Index": 0.40,
        "LEGATRUU Index": 0.20
    }
}

# Check that the weights of each target sum to 1
for target_name, weights in new_target_definitions.items():
    total_weight = sum(weights.values())
    
    if not np.isclose(total_weight, 1.0):
        raise ValueError(
            f"Weights for {target_name} do not sum to 1. Current sum: {total_weight:.4f}"
        )

print("All target weights sum to 1.")

# Display the composition of the new synthetic targets
new_target_composition = pd.DataFrame(new_target_definitions).fillna(0).T

new_target_composition_display = new_target_composition.copy()

for col in new_target_composition_display.columns:
    new_target_composition_display[col] = new_target_composition_display[col].map(
        lambda x: f"{x * 100:.0f}%"
    )

print("New synthetic targets - composition:")
display(new_target_composition_display)

In [ ]:
# Build weekly returns for each new synthetic target
new_targets_returns = pd.DataFrame(index=component_returns.index)

for target_name, weights in new_target_definitions.items():
    target_series = pd.Series(0.0, index=component_returns.index)
    
    for component, weight in weights.items():
        target_series += component_returns[component] * weight
    
    new_targets_returns[target_name] = target_series

# Align with the same dates used for the main synthetic target and futures
new_targets_returns = new_targets_returns.loc[common_dates]

print("New synthetic targets - weekly returns preview:")
display(new_targets_returns.head())



In [ ]:
# Compare main synthetic target with the new synthetic targets
all_synthetic_targets = pd.concat(
    [
        target_returns_aligned.rename("Principal Target"),
        new_targets_returns
    ],
    axis=1
)

all_targets_stats_raw = pd.DataFrame({
    "Ann. Return": all_synthetic_targets.mean() * 52,
    "Ann. Volatility": all_synthetic_targets.std() * np.sqrt(52),
    "Sharpe Ratio": (
        all_synthetic_targets.mean() * 52
    ) / (
        all_synthetic_targets.std() * np.sqrt(52)
    ),
    "Max Drawdown": all_synthetic_targets.apply(max_drawdown),
    "Weekly VaR 95%": all_synthetic_targets.quantile(0.05),
    "Weekly Expected Shortfall 95%": all_synthetic_targets.apply(
        lambda x: x[x <= x.quantile(0.05)].mean()
    ),
    "Skewness": all_synthetic_targets.skew(),
    "Kurtosis": all_synthetic_targets.kurtosis()
})

all_targets_stats = all_targets_stats_raw.copy()

percentage_cols = [
    "Ann. Return",
    "Ann. Volatility",
    "Max Drawdown",
    "Weekly VaR 95%",
    "Weekly Expected Shortfall 95%"
]

for col in percentage_cols:
    all_targets_stats[col] = all_targets_stats[col].map(lambda x: f"{x * 100:.2f}%")

for col in ["Sharpe Ratio", "Skewness", "Kurtosis"]:
    all_targets_stats[col] = all_targets_stats[col].map(lambda x: f"{x:.2f}")

print("Comparison of synthetic targets:")
display(all_targets_stats)

In [ ]:
# Plot cumulative performance of all synthetic targets
all_targets_cumulative = (1 + all_synthetic_targets).cumprod()

plt.figure(figsize=(14, 7))

for col in all_targets_cumulative.columns:
    plt.plot(
        all_targets_cumulative.index,
        all_targets_cumulative[col],
        linewidth=2,
        label=col
    )

plt.title("Cumulative Performance of Synthetic Targets")
plt.xlabel("Date")
plt.ylabel("Cumulative growth of 1 unit invested")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Preliminary replicability check: correlations with futures
replicability_rows = []

for target_name in all_synthetic_targets.columns:
    target_series = all_synthetic_targets[target_name]
    
    corr_with_futures = futures_returns.corrwith(target_series).sort_values(
        key=lambda x: x.abs(),
        ascending=False
    )
    
    top_future = corr_with_futures.index[0]
    
    replicability_rows.append({
        "Target": target_name,
        "Top Future": top_future,
        "Top Future Description": variable_info.get(top_future, top_future),
        "Top Correlation": corr_with_futures.iloc[0],
        "Average Top 3 Abs Corr": corr_with_futures.abs().head(3).mean()
    })

replicability_summary = pd.DataFrame(replicability_rows)

replicability_summary["Top Correlation"] = replicability_summary["Top Correlation"].map(
    lambda x: f"{x:.3f}"
)
replicability_summary["Average Top 3 Abs Corr"] = replicability_summary[
    "Average Top 3 Abs Corr"
].map(lambda x: f"{x:.3f}")

print("Preliminary replicability summary:")
display(replicability_summary)

## 4.1 Autocorrelation check across different synthetic targets

In [ ]:
# Ljung-Box test across all synthetic targets
synthetic_targets_ljung_box = []

for target_name in all_synthetic_targets.columns:
    series = all_synthetic_targets[target_name].dropna()
    
    lb_test = acorr_ljungbox(
        series,
        lags=[5, 10, 15, 20],
        return_df=True
    )
    
    for lag in [5, 10, 15, 20]:
        synthetic_targets_ljung_box.append({
            "Target": target_name,
            "Lag": lag,
            "Ljung-Box statistic": lb_test.loc[lag, "lb_stat"],
            "p-value": lb_test.loc[lag, "lb_pvalue"]
        })

synthetic_targets_ljung_box = pd.DataFrame(synthetic_targets_ljung_box)

synthetic_targets_ljung_box_display = synthetic_targets_ljung_box.copy()

synthetic_targets_ljung_box_display["Ljung-Box statistic"] = synthetic_targets_ljung_box_display[
    "Ljung-Box statistic"
].map(lambda x: f"{x:.2f}")

synthetic_targets_ljung_box_display["p-value"] = synthetic_targets_ljung_box_display[
    "p-value"
].map(lambda x: f"{x:.4f}")

print("Ljung-Box test across synthetic targets:")
display(synthetic_targets_ljung_box_display)

## ARCH test across all synthetic targets


In [ ]:
arch_targets_results = []

for target_name in all_synthetic_targets.columns:
    r = all_synthetic_targets[target_name].dropna().values
    
    lm_stat, lm_pval, _, _ = het_arch(r, nlags=5)
    
    arch_targets_results.append({
        "Target": target_name,
        "ARCH LM Statistic": lm_stat,
        "p-value": lm_pval,
        "Volatility Clustering": "Present" if lm_pval < 0.05 else "Not significant"
    })

arch_targets_results = pd.DataFrame(arch_targets_results)

arch_targets_display = arch_targets_results.copy()

arch_targets_display["ARCH LM Statistic"] = arch_targets_display[
    "ARCH LM Statistic"
].map(lambda x: f"{x:.2f}")

arch_targets_display["p-value"] = arch_targets_display[
    "p-value"
].map(lambda x: f"{x:.2e}")

print("ARCH test across synthetic targets:")
display(arch_targets_display)

## 6.8 PCA on futures returns

In [ ]:
# PCA on futures returns
pca = PCA()
pca.fit(futures_returns.dropna())

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.bar(range(1, len(explained) + 1), explained * 100, alpha=0.8)
ax1.plot(range(1, len(explained) + 1), explained * 100, "o-", linewidth=1.5)
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained variance (%)")
ax1.set_title("Scree Plot - PCA on Futures Returns")
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(cumulative) + 1), cumulative * 100, "o-", linewidth=2)
ax2.axhline(80, linestyle="--", alpha=0.7, label="80% threshold")
ax2.axhline(90, linestyle="--", alpha=0.7, label="90% threshold")
ax2.set_xlabel("Number of components")
ax2.set_ylabel("Cumulative explained variance (%)")
ax2.set_title("Cumulative Explained Variance - Futures PCA")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

loadings = pd.DataFrame(
    pca.components_[:5].T,
    index=[variable_info.get(c, c) for c in futures_returns.columns],
    columns=[f"PC{i+1}" for i in range(5)]
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    loadings,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5
)

plt.title("PCA Loadings - First 5 Principal Components")
plt.tight_layout()
plt.show()

n_80 = np.searchsorted(cumulative, 0.80) + 1
n_90 = np.searchsorted(cumulative, 0.90) + 1

print(f"PCs needed to explain >= 80% of variance: {n_80}")
print(f"PCs needed to explain >= 90% of variance: {n_90}")
print(f"Variance explained by first 3 PCs      : {cumulative[2] * 100:.1f}%")

## 6.9 Deliverable summary

In [ ]:
# Deliverable summary

print("=" * 60)
print("DELIVERABLE SUMMARY")
print("=" * 60)

print("target_returns_aligned")
print(f"  Type   : {type(target_returns_aligned).__name__}")
print(f"  Shape  : {target_returns_aligned.shape}")
print(f"  Name   : '{target_returns_aligned.name}'")
print(f"  Range  : {target_returns_aligned.index[0].date()} → {target_returns_aligned.index[-1].date()}")

print()

print("futures_returns")
print(f"  Type   : {type(futures_returns).__name__}")
print(f"  Shape  : {futures_returns.shape}")
print(f"  Columns: {futures_returns.columns.tolist()}")
print(f"  Range  : {futures_returns.index[0].date()} → {futures_returns.index[-1].date()}")

print()

print(f"Indices aligned: {(target_returns_aligned.index == futures_returns.index).all()}")
print("=" * 60)